# Task #49 — Huấn luyện Logistic Regression, Random Forest, XGBoost trên tập train (Story #9)

Đọc `orders_features_train.csv` (đã split ở Task #46, đã lọc `is_delayed` NA và các đơn thiếu đặc trưng), huấn luyện 3 mô hình baseline áp dụng `class_weight`/`scale_pos_weight` đã tính sẵn ở Task #47 (không resample dữ liệu). Đo thời gian huấn luyện từng mô hình và lưu lại — dùng cho bảng so sánh ở Task #50. Không đụng đến tập test ở notebook này.

In [1]:
import time
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

df = pd.read_csv("../data/processed/orders_features_train.csv", low_memory=False)

df["is_delayed"] = df["is_delayed"].astype(bool)

bool_cols = ["payment_has_boleto", "payment_has_credit_card", "payment_has_debit_card",
             "payment_has_not_defined", "payment_has_voucher", "items_multi_seller"]
for col in bool_cols:
    df[col] = df[col].astype("boolean")

X_train = df.drop(columns=["order_id", "is_delayed"])
y_train = df["is_delayed"].astype(int)

print("X_train:", X_train.shape, " y_train:", y_train.shape)

X_train: (77156, 75)  y_train: (77156,)


## 1. Trọng số xử lý mất cân bằng (đã tính sẵn ở Task #47)

Dùng lại trực tiếp, không tính lại.

In [2]:
class_weight_dict = {False: 0.5441568516820651, True: 6.161635521482191}
scale_pos_weight = 11.3233

# sklearn nhan key bool True/False lam class_weight cho y int 0/1
class_weight_sklearn = {0: class_weight_dict[False], 1: class_weight_dict[True]}

print("class_weight (Logistic Regression / Random Forest):", class_weight_sklearn)
print("scale_pos_weight (XGBoost):", scale_pos_weight)

class_weight (Logistic Regression / Random Forest): {0: 0.5441568516820651, 1: 6.161635521482191}
scale_pos_weight (XGBoost): 11.3233


## 2. Huấn luyện 3 mô hình baseline

`random_state=42` cho cả 3 mô hình, khớp seed đã dùng ở Task #46 (train_test_split) — chưa có quy ước seed chính thức toàn dự án, chọn theo tự đánh giá kỹ thuật để nhất quán trong phạm vi Story #9.

In [3]:
training_times = {}

start = time.perf_counter()
log_reg = LogisticRegression(class_weight=class_weight_sklearn, max_iter=1000, random_state=42, n_jobs=-1)
log_reg.fit(X_train, y_train)
training_times["logistic_regression"] = time.perf_counter() - start

print(f"Logistic Regression: {training_times['logistic_regression']:.2f}s")

D:\Project Management\delivery-performance-intelligence\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Logistic Regression: 31.86s


D:\Project Management\delivery-performance-intelligence\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [4]:
start = time.perf_counter()
rand_forest = RandomForestClassifier(class_weight=class_weight_sklearn, random_state=42, n_jobs=-1)
rand_forest.fit(X_train, y_train)
training_times["random_forest"] = time.perf_counter() - start

print(f"Random Forest: {training_times['random_forest']:.2f}s")

Random Forest: 3.72s


In [5]:
start = time.perf_counter()
xgb = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, n_jobs=-1, eval_metric="logloss")
xgb.fit(X_train, y_train)
training_times["xgboost"] = time.perf_counter() - start

print(f"XGBoost: {training_times['xgboost']:.2f}s")

XGBoost: 3.10s


## 3. Lưu mô hình đã huấn luyện + thời gian huấn luyện

Lưu vào `models/` (mới) bằng `joblib` — Task #50 sẽ load lại để đánh giá trên tập test, không huấn luyện lại.

In [6]:
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

joblib.dump(log_reg, models_dir / "logistic_regression.pkl")
joblib.dump(rand_forest, models_dir / "random_forest.pkl")
joblib.dump(xgb, models_dir / "xgboost.pkl")

with open(models_dir / "training_times.json", "w", encoding="utf-8") as f:
    json.dump(training_times, f, indent=2)

print("Da luu 3 model va training_times.json vao models/")
print(training_times)

Da luu 3 model va training_times.json vao models/
{'logistic_regression': 31.861822000006214, 'random_forest': 3.7170604999992065, 'xgboost': 3.0953199000068707}
